# Step 04.3 — Non-linearity：为什么 ReLU / GELU 必须出现

前两节已经知道：Linear 可以做特征混合，参数可以训练。

现在出现一个关键问题：**如果我们把很多 Linear 一层层堆起来，会不会自然变成一个很强的深度网络？**

答案是：**如果中间没有非线性，很多层 Linear 仍然可以合并成一次 affine transformation。**

本节只解决三件事：

1. 为什么 `Linear → Linear` 会塌缩成一个 Linear/affine map；
2. activation function 如何打破这种塌缩；
3. ReLU 与 GELU 的行为，以及为什么 MiniGPT 选择 GELU。

## 1. 两层 Linear 为什么仍然只是一次 affine transformation？

先忽略 bias，第一层：

```text
h = x W1ᵀ
```

第二层：

```text
y = h W2ᵀ
```

把第一式代入第二式：

```text
y = (x W1ᵀ) W2ᵀ
```

矩阵乘法满足结合律，所以：

```text
y = x (W1ᵀ W2ᵀ)
```

令 `Wnewᵀ = W1ᵀ W2ᵀ`，就得到：

```text
y = x Wnewᵀ
```

所以两层 Linear 在函数表达能力上仍然等价于一层 Linear。

In [ ]:
import mlx.core as mx

x = mx.array([[2.0, -1.0]])

W1 = mx.array([
    [1.0, 2.0],
    [0.0, 1.0],
    [1.0, -1.0],
])

W2 = mx.array([
    [1.0, 0.0, 2.0],
    [-1.0, 1.0, 0.0],
])

h = x @ W1.T
y_two_layers = h @ W2.T

Wnew_T = W1.T @ W2.T
y_one_layer = x @ Wnew_T

mx.eval(y_two_layers, y_one_layer)

print("two Linear layers:", y_two_layers)
print("collapsed Linear  :", y_one_layer)
print("same:", bool(mx.allclose(y_two_layers, y_one_layer)))

如果看到 `same: True`，说明两层确实可以合并。

即使加入 bias，也仍然可以合并：

```text
h = xW1ᵀ + b1
y = hW2ᵀ + b2
```

代入：

```text
y = x(W1ᵀW2ᵀ) + (b1W2ᵀ + b2)
```

所以仍然只是一次新的 `xWᵀ+b`。

也就是说，单纯堆叠 affine transformation，不会自动得到更丰富的函数族。

## 2. Activation function：打破可合并性

现在把网络改成：

```text
h = Linear1(x)
h = activation(h)
y = Linear2(h)
```

也就是：

```text
y = W2 · activation(W1x + b1) + b2
```

只要 activation 是非线性的，中间这一步就不能被简单吸收到一个固定矩阵里。

这就是激活函数最基础的意义：**让多层网络真正比一层 Linear 更有表达能力。**

### 什么叫非线性？

例如：

```text
f(x)=2x+3
```

始终是一条直线。

而：

```text
max(0,x)
x²
sin(x)
```

都不能被一个固定的 `ax+b` 在所有输入上完全表示，所以属于非线性函数。

## 3. ReLU：简单的非线性激活

ReLU = Rectified Linear Unit：

```text
ReLU(x) = max(0, x)
```

所以：

```text
[-2, -1, 0, 1, 2]
        ↓
[ 0,  0, 0, 1, 2]
```

它在 0 处发生折角，因此整体不能被一个单一 affine map 表示。

In [ ]:
import mlx.nn as nn

values = mx.array([-2.0, -1.0, 0.0, 1.0, 2.0])
relu = nn.ReLU()

relu_out = relu(values)
mx.eval(relu_out)

print("input:", values)
print("ReLU :", relu_out)

## 4. GELU：平滑地决定保留多少输入

GELU = Gaussian Error Linear Unit。

一个很有用的定义是：

```text
GELU(x) = x · Φ(x)
```

`Φ(x)` 是标准正态分布的累积分布函数（Gaussian CDF）。现在不需要学概率论细节，只需知道它是 0 到 1 之间、随 x 平滑变化的系数。

直觉：

```text
x 很负 -> Φ(x) 接近 0
x = 0  -> Φ(x) = 0.5
x 很正 -> Φ(x) 接近 1
```

所以 GELU 可以理解成：**根据输入值本身的大小，平滑决定保留这个输入的多少。**

In [ ]:
gelu = nn.GELU()
gelu_out = gelu(values)
mx.eval(gelu_out)

print("input:", values)
print("ReLU :", relu_out)
print("GELU :", gelu_out)

GELU 不会像 ReLU 那样把所有负输入直接设为 0，而是保留一些小的负输出；正输入也不是立刻 100% 原样通过，而是平滑接近输入本身。

## 5. 直接验证：加入 GELU 后，原来的矩阵合并公式失效

没有 GELU：

```text
x -> W1 -> W2
```

可以合并。

加入 GELU：

```text
x -> W1 -> GELU -> W2
```

原来的 `Wnewᵀ = W1ᵀW2ᵀ` 已经不能代表整个网络。

In [ ]:
h_linear = x @ W1.T
h_nonlinear = gelu(h_linear)
y_with_gelu = h_nonlinear @ W2.T

y_collapsed_old_way = x @ (W1.T @ W2.T)

mx.eval(y_with_gelu, y_collapsed_old_way)

print("with GELU             :", y_with_gelu)
print("old collapsed formula :", y_collapsed_old_way)
print("same:", bool(mx.allclose(y_with_gelu, y_collapsed_old_way)))

一般会看到 `same: False`。

这只是在证明：原来那种直接把两组矩阵乘起来的合并方法已经失效。

更深层的理论是：带非线性的多层网络能够表示远比单一 affine map 丰富得多的函数族。

## 为什么我们的 MiniGPT 用 GELU？

经典 GPT 风格 Transformer 的 feed-forward network 常使用 GELU，因此我们的教学版 MiniGPT 也采用它，便于贴近这条历史路线。

但不要形成两个错误结论：

1. GELU 不是现代所有 LLM 的唯一标准；很多现代 LLM 使用 SwiGLU 等 gated activation；
2. GELU 也不是因为“更平滑所以一定训练更好”这么简单。训练效果由架构、初始化、归一化、优化器等多因素共同决定。

这里最重要的基础作用仍然是：**引入非线性。**

### 4.3 最终结论

- 多层 Linear/affine 层如果中间没有非线性，可以合并成一次新的 affine transformation；
- 所以单纯增加 Linear 层数不会自动增加函数表达能力；
- activation function 打破这种可合并性；
- ReLU：`max(0,x)`，简单、分段线性；
- GELU：`x·Φ(x)`，对输入做平滑门控；
- MiniGPT 使用 GELU，是为了贴近经典 GPT/Transformer 的 feed-forward 设计；
- 下一节 4.4 会第一次把这些积木组合成真正的 MLP：`C → 4C → GELU → C`。